# Unificação e Filtragem da Base GERCON — Município Executante (Porto Alegre - RS)

Este notebook realiza a leitura automatizada, junção de todos os datasets anuais (`.csv`) do sistema **GERCON** (2018 a 2026) presentes na pasta do Google Drive, aplica o filtro pelo **Município Executante: Porto Alegre** e padroniza a inclusão do **Código IBGE de Porto Alegre (`4314902` / `431490`)**.

---

### ⚙️ Estratégia de Processamento:
1. **Montagem do Google Drive**: Acesso direto aos arquivos da tese/doutorado.
2. **Leitura Eficiente**: Processamento individual arquivo a arquivo para manter o uso de memória RAM baixo no Colab.
3. **Detecção Flexível**: Tratamento automático de delimitador (`;` ou `,`) e codificação (`utf-8` ou `latin-1`).
4. **Filtragem por Município Executante**: Filtro por "PORTO ALEGRE" e/ou código IBGE na coluna correspondente (`MUNICIPIO_SOLICITANTE`).
5. **Inclusão do Código IBGE**: Criação e padronização explícita das colunas do IBGE (`4314902` de 7 dígitos e `431490` de 6 dígitos).
6. **Exportação**: Geração de um novo arquivo consolidado `.csv` gravado diretamente no seu Google Drive.


## 1. 🔗 Montagem do Google Drive


In [22]:
# Instalar/atualizar o gdown para baixar pastas do Google Drive de forma recursiva
# !pip install --upgrade gdown

import os
import glob
import unicodedata
import gc
import pandas as pd
import numpy as np
import gdown
from google.colab import files

# URL da pasta compartilhada do Google Drive
folder_url = 'https://drive.google.com/drive/folders/1reZ84EmgnP38uamfnvU57BwQ3WUO2NL_?usp=sharing'

# Definir diretório local na seção do Colab
caminho_gercon = '/content/gercon'
os.makedirs(caminho_gercon, exist_ok=True)

# Fazer o download de todo o conteúdo para a seção (removido parâmetro incompatível 'remaining_ok')
gdown.download_folder(url=folder_url, output=caminho_gercon, quiet=False)

Retrieving folder contents


Processing file 12K--XYD_VvCMStChmLbMYOQxBkOgDINh gercon_solicitacoes_2018.csv
Processing file 1oA0y5ZnU0PFNq9oKU7C7aQK9iK1kkJgU gercon_solicitacoes_2019.csv
Processing file 16HiwoBSg7GfdxLfmNApf1-TiGI2jMmxu gercon_solicitacoes_2020.csv
Processing file 1B9rL-Fa_h9iN0Cx8T5FBHpS31xwK7VwM gercon_solicitacoes_2021.csv
Processing file 1Gs6acana4x94h-FyEyaCxxp2yKKGMTKJ gercon_solicitacoes_2022.csv
Processing file 1MShjQkfZgmUvLTJ6RzdSHqTBaTiiDoY4 gercon_solicitacoes_2023.csv
Processing file 1rbvL45lYTCQg5TnoPC2bz6gqzbuxS83x gercon_solicitacoes_2024 (1).csv
Processing file 1Al_TdTIc9sL5xXkTDTZ3HSB0QtgI52y_ gercon_solicitacoes_2025.csv
Processing file 1Q-lwXWE1xJJdyaMncbelNBt4_zN-eAJJ gercon_solicitacoes_2026.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed


KeyboardInterrupt: 

## 2. 📂 Localização dos Arquivos GERCON


In [ ]:
# Caminho da pasta baixada diretamente na seção local do Colab
caminho_gercon = '/content/gercon'

print(f"Diretório selecionado: {caminho_gercon}")

# Listar todos os arquivos CSV brutos baixados na pasta local
arquivos_csv = sorted([
    f for f in glob.glob(os.path.join(caminho_gercon, "*.csv"))
    if not os.path.basename(f).startswith("gercon_solicitacoes_executante_poa")
    and not os.path.basename(f).startswith("dados_poa")
])

print(f"\nTotal de arquivos CSV encontrados: {len(arquivos_csv)}")
for arq in arquivos_csv:
    tam_mb = os.path.getsize(arq) / (1024 * 1024)
    print(f"  • {os.path.basename(arq)} ({tam_mb:.1f} MB)")

In [ ]:
# Carregar o arquivo de amostra de 2020 diretamente da pasta local
sample = pd.read_csv("/content/gercon/gercon_solicitacoes_2020.csv", dtype=str, sep=";")

In [ ]:
# Visualizar as informações estruturais do DataFrame amostral de 2020
sample.info()

In [ ]:
# 1. Deixar a coluna 'municipiosolicitante' com todas as letras maiúsculas
sample['municipiosolicitante'] = sample['municipiosolicitante'].str.upper()

# 2. Contar a frequência relativa dos municípios que começam com 'P'
#municipios_com_p = sample[sample['municipiosolicitante'].str.startswith('P')]
frequencia_relativa_p = sample['municipiosolicitante'].value_counts(normalize=True)

print("Frequência relativa dos municípios que começam com 'P' na coluna 'municipiosolicitante':")
display(frequencia_relativa_p)

## 3. 🛠️ Funções de Leitura, Padronização e Filtro

- **Código IBGE de Porto Alegre**: `4314902` (7 dígitos) e `431490` (6 dígitos).
- Tratamento para garantir a integridade dos tipos e caracteres com acentuação.


In [ ]:
# Constantes oficiais do IBGE para Porto Alegre / RS
IBGE_POA_7D = "4314902"
IBGE_POA_6D = "431490"
NOME_POA = "PORTO ALEGRE"

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8").strip().upper()

def carregar_e_filtrar_arquivo(caminho_arquivo):
    nome_base = os.path.basename(caminho_arquivo)
    print(f"\n➡️ Processando: {nome_base}...")

    df = None
    # Identificar separador e codificação
    for encoding in ["utf-8", "latin-1", "iso-8859-1"]:
        for sep in [";", ","]:
            try:
                sample = pd.read_csv(caminho_arquivo, sep=sep, encoding=encoding, nrows=10, dtype=str)
                if len(sample.columns) > 5:
                    df = pd.read_csv(caminho_arquivo, sep=sep, encoding=encoding, dtype=str, low_memory=False)
                    print(f"   ✓ Lido com sucesso (sep='{sep}', enc='{encoding}', total={len(df):,} linhas)")
                    break
            except Exception:
                continue
        if df is not None:
            break

    if df is None:
        raise ValueError(f"Não foi possível abrir o arquivo: {caminho_arquivo}")

    # Localizar a coluna do município executante
    colunas_map = {col.upper().strip(): col for col in df.columns}

    col_executante = None
    for cand in ["MUNICIPIOSOLICITANTE", "municipiosolicitante"]:
        if cand in colunas_map:
            col_executante = colunas_map[cand]
            break

    if col_executante is None:
        for c in df.columns:
            if "SOLICITANTE" in c.upper() and ("MUN" in c.upper() or "CIDADE" in c.upper()):
                col_executante = c
                break

    if col_executante is None:
        print(f"   ⚠️ Atenção: Coluna de município solicitante não encontrada em {nome_base}!")
        return pd.DataFrame()

    # Aplicar filtro por Porto Alegre
    serie_norm = df[col_executante].fillna("").apply(normalizar_texto)
    filtro_poa = (
        (serie_norm == NOME_POA) |
        (df[col_executante].str.strip() == IBGE_POA_7D) |
        (df[col_executante].str.strip() == IBGE_POA_6D) |
        (serie_norm.str.contains("PORTO ALEGRE"))
    )

    df_filtrado = df[filtro_poa].copy()

    # Inserir/Padronizar os códigos IBGE de Porto Alegre
    df_filtrado["COD_IBGE_MUNICIPIO_EXECUTANTE_7D"] = IBGE_POA_7D
    df_filtrado["COD_IBGE_MUNICIPIO_EXECUTANTE_6D"] = IBGE_POA_6D
    df_filtrado["UF_EXECUTANTE"] = "RS"
    df_filtrado["ARQUIVO_ORIGEM"] = nome_base

    print(f"   ✓ Linhas retidas (POA Executante): {len(df_filtrado):,} ({len(df_filtrado)/len(df)*100:.2f}% do arquivo)")

    del df
    gc.collect()
    return df_filtrado



## 4. 🔄 Processamento e Junção de Todos os Anos


In [ ]:
dfs_filtrados = []

for caminho in arquivos_csv:
    df_parcial = carregar_e_filtrar_arquivo(caminho)
    if not df_parcial.empty:
        dfs_filtrados.append(df_parcial)

if dfs_filtrados:
    df_gercon_poa_final = pd.concat(dfs_filtrados, ignore_index=True)
    print("\n" + "="*60)
    print("🎉 UNIFICAÇÃO CONCLUÍDA COM SUCESSO!")
    print(f"• Total consolidado de linhas (POA solicitante): {len(df_gercon_poa_final):,}")
    print(f"• Total de colunas: {df_gercon_poa_final.shape[1]}")
    print("="*60)
else:
    print("❌ Nenhum dado foi filtrado.")



## 5. 📊 Diagnóstico e Validação do Dataset Consolidado


In [ ]:
# Distribuição por ano/arquivo de origem
print("📌 Quantidade de solicitações por arquivo:")
print(df_gercon_poa_final["ARQUIVO_ORIGEM"].value_counts().sort_index())

# Visualizar primeiras linhas
print("\n📌 Amostra dos dados consolidados:")
display(df_gercon_poa_final.head(10))

# Checagem das colunas IBGE criadas
print("\n📌 Verificação dos Códigos IBGE de Porto Alegre:")
print(df_gercon_poa_final[["COD_IBGE_MUNICIPIO_EXECUTANTE_7D", "COD_IBGE_MUNICIPIO_EXECUTANTE_6D", "UF_EXECUTANTE"]].drop_duplicates())



## 6. 💾 Exportação para Arquivo .CSV Consolidado no Google Drive


In [ ]:
# Garantir que df_gercon_poa_final esteja definido no escopo atual
if 'df_gercon_poa_final' not in locals() and 'df_gercon_poa_final' not in globals():
    if 'dfs_filtrados' in locals() or 'dfs_filtrados' in globals():
        if dfs_filtrados:
            df_gercon_poa_final = pd.concat(dfs_filtrados, ignore_index=True)
        else:
            raise NameError("A lista 'dfs_filtrados' está vazia. Execute a unificação primeiro.")
    else:
        raise NameError("A variável 'df_gercon_poa_final' não foi definida no ambiente.")

# Nome do arquivo de saída consolidado em formato Parquet
nome_arquivo_saida = "gercon_solicitacoes_executante_porto_alegre_2018_2025.parquet"
caminho_saida = os.path.join(caminho_gercon, nome_arquivo_saida)

print(f"Salvando dataset consolidado em formato Parquet em:\n{caminho_saida} ...")

# Exportar como Parquet usando compressão snappy para máxima eficiência
df_gercon_poa_final.to_parquet(caminho_saida, index=False, compression="snappy")

tamanho_mb = os.path.getsize(caminho_saida) / (1024 * 1024)
print(f"\n✅ Arquivo Parquet gerado com sucesso localmente na sessão!")
print(f"📦 Tamanho do arquivo compactado: {tamanho_mb:.2f} MB")
print(f"📁 Caminho de acesso: {caminho_saida}")

In [ ]:
# Caminho do arquivo parquet gerado
caminho_parquet = "/content/gercon/gercon_solicitacoes_executante_porto_alegre_2018_2025.parquet"

# Iniciar o download automático no seu navegador
files.download(caminho_parquet)